# Update Existing Residential Units — Parcel_History_Attributed

**Goal:** Compare CSV source data against `SDE.Parcel_History_Attributed` in the enterprise geodatabase, validate differences, identify missing APNs, then update the `Residential_Units` field.

**Source CSV:** `data/raw_data/ExistingResidential_2012_2025_unstacked.csv`  
**Target FC:** `SDE.Parcels\SDE.Parcel_History_Attributed` (Vector.sde)  
**Service:** https://maps.trpa.org/server/rest/services/Existing_Development/MapServer/2

## 1. Setup

In [ ]:
import os
import pathlib
import warnings

import pandas as pd
import numpy as np
import arcpy

warnings.filterwarnings('ignore')

# ── Paths ──────────────────────────────────────────────────────────────────
local_path = pathlib.Path().absolute()

# SDE connection files — try both common locations
for _candidate in [
    pathlib.Path(r"F:\GIS\DB_CONNECT\Vector.sde"),
    pathlib.Path(r"F:\GIS\PARCELUPDATE\Workspace\Vector.sde"),
]:
    if _candidate.exists():
        sdeBase = _candidate
        break
else:
    raise FileNotFoundError("Cannot find Vector.sde — update the path above.")

print(f"Using SDE: {sdeBase}")

# Feature class path
fc_path = str(sdeBase / "sde.SDE.Parcels" / "sde.SDE.Parcel_History_Attributed")
print(f"Feature class: {fc_path}")

# CSV
csv_path = local_path / "data" / "raw_data" / "ExistingResidential_2012_2025_unstacked.csv"
print(f"CSV: {csv_path}")
assert csv_path.exists(), f"CSV not found: {csv_path}"

# arcpy settings
arcpy.env.workspace = 'memory'
arcpy.env.overwriteOutput = True

## 2. Load and Reshape CSV

The CSV is **wide format** (one row per APN, year columns).  
We melt it to **long format** (one row per APN × Year) to match the feature class structure.

In [ ]:
# Read CSV — drop any completely empty columns (the extra trailing columns in the file)
df_wide = pd.read_csv(csv_path)

# Keep only APN + year columns ("YYYY Final" pattern)
year_cols = [c for c in df_wide.columns if str(c).strip().endswith('Final')]
df_wide   = df_wide[['APN'] + year_cols].copy()

# Strip whitespace from APN
df_wide['APN'] = df_wide['APN'].astype(str).str.strip()

print(f"CSV rows (APNs): {len(df_wide):,}")
print(f"Year columns   : {year_cols}")
df_wide.head()

In [ ]:
# Melt to long format
df_csv = df_wide.melt(id_vars='APN', var_name='Year_Label', value_name='Residential_Units_CSV')

# Extract integer year from "YYYY Final"
df_csv['Year'] = df_csv['Year_Label'].str.extract(r'(\d{4})').astype(int)
df_csv = df_csv.drop(columns='Year_Label')

# Drop rows where Residential_Units is null (APN not present that year)
df_csv = df_csv.dropna(subset=['Residential_Units_CSV'])
df_csv['Residential_Units_CSV'] = df_csv['Residential_Units_CSV'].astype(int)

print(f"Long-format rows: {len(df_csv):,}")
print(f"Unique APNs     : {df_csv['APN'].nunique():,}")
print(f"Years covered   : {sorted(df_csv['Year'].unique())}")
df_csv.head(10)

## 3. Load Feature Class into Pandas

Pull the relevant fields from `SDE.Parcel_History_Attributed` using arcpy's Spatially Enabled DataFrame.

In [ ]:
# First, inspect the feature class schema to confirm field names
fields = arcpy.ListFields(fc_path)
field_names = [f.name for f in fields]
print("Feature class fields:")
for f in fields:
    print(f"  {f.name:<40} {f.type}")

In [ ]:
# ── ADJUST THESE if the actual field names differ from what's shown above ──
FC_APN_FIELD   = 'APN'               # APN field in the feature class
FC_YEAR_FIELD  = 'YEAR'              # Year field in the feature class
FC_RES_FIELD   = 'Residential_Units' # Field we want to update
# ──────────────────────────────────────────────────────────────────────────

# Confirm fields exist
for fld in [FC_APN_FIELD, FC_YEAR_FIELD, FC_RES_FIELD]:
    assert fld in field_names, f"Field '{fld}' NOT FOUND in feature class. Available: {field_names}"
    print(f"  ✓ {fld}")

In [ ]:
# Load feature class — APN, Year, Residential_Units only (no geometry needed for comparison)
read_fields = [FC_APN_FIELD, FC_YEAR_FIELD, FC_RES_FIELD, 'OID@']

rows = []
with arcpy.da.SearchCursor(fc_path, read_fields) as cursor:
    for row in cursor:
        rows.append({
            'APN'              : str(row[0]).strip() if row[0] else None,
            'Year'             : int(row[1]) if row[1] is not None else None,
            'Residential_Units': row[2],
            'OID'              : row[3],
        })

df_fc = pd.DataFrame(rows)

# Filter to the years present in our CSV
csv_years = sorted(df_csv['Year'].unique())
df_fc_filtered = df_fc[df_fc['Year'].isin(csv_years)].copy()

print(f"FC total rows        : {len(df_fc):,}")
print(f"FC rows (2012–2025)  : {len(df_fc_filtered):,}")
print(f"FC unique APNs       : {df_fc_filtered['APN'].nunique():,}")
df_fc_filtered.head(10)

## 4. Validate — Compare CSV vs Feature Class

In [ ]:
# ── APN sets ───────────────────────────────────────────────────────────────
csv_apns = set(df_csv['APN'].unique())
fc_apns  = set(df_fc_filtered['APN'].dropna().unique())

missing_from_fc  = csv_apns - fc_apns   # in CSV but NOT in FC → need to add
missing_from_csv = fc_apns  - csv_apns  # in FC but NOT in CSV → FYI only

print(f"APNs in CSV              : {len(csv_apns):,}")
print(f"APNs in FC (2012–2025)   : {len(fc_apns):,}")
print(f"APNs in CSV missing from FC : {len(missing_from_fc):,}  ← need geometry lookup")
print(f"APNs in FC not in CSV       : {len(missing_from_csv):,}")

In [ ]:
# ── APNs missing from FC ───────────────────────────────────────────────────
df_missing = df_csv[df_csv['APN'].isin(missing_from_fc)].copy()

print(f"\nAPNs missing from feature class ({len(missing_from_fc):,}):")
print(sorted(missing_from_fc)[:50], "..." if len(missing_from_fc) > 50 else "")

print(f"\nRows affected (APN × Year): {len(df_missing):,}")
df_missing.sort_values(['APN','Year']).head(20)

In [ ]:
# ── Value differences for APNs that ARE in both ────────────────────────────
# Merge on APN + Year
df_compare = df_csv.merge(
    df_fc_filtered[['APN','Year','Residential_Units','OID']],
    on=['APN','Year'],
    how='inner'
)

df_compare['values_match'] = (
    df_compare['Residential_Units_CSV'] == df_compare['Residential_Units']
)

df_diffs = df_compare[~df_compare['values_match']].copy()

print(f"Matched APN×Year rows    : {len(df_compare):,}")
print(f"Rows with value mismatch : {len(df_diffs):,}")
print(f"Rows already matching    : {df_compare['values_match'].sum():,}")

if len(df_diffs):
    df_diffs[['APN','Year','Residential_Units_CSV','Residential_Units']].rename(
        columns={'Residential_Units':'Residential_Units_FC'}
    ).sort_values(['APN','Year']).head(30)

In [ ]:
# ── Summary table ──────────────────────────────────────────────────────────
summary = pd.DataFrame({
    'Category': [
        'APNs in CSV',
        'APNs in FC (filtered to CSV years)',
        'APNs missing from FC (need geometry)',
        'APNs in FC not in CSV',
        'APN×Year rows matched',
        'APN×Year value mismatches (need update)',
        'APN×Year already correct',
    ],
    'Count': [
        len(csv_apns),
        len(fc_apns),
        len(missing_from_fc),
        len(missing_from_csv),
        len(df_compare),
        len(df_diffs),
        int(df_compare['values_match'].sum()),
    ]
})
summary

## 5. Export Validation Results

Save the differences and missing APNs to CSV so we can review before making any edits.

In [ ]:
out_dir = local_path / "data" / "processed_data"
out_dir.mkdir(parents=True, exist_ok=True)

# Missing APNs
missing_out = out_dir / "ResUnits_APNs_missing_from_FC.csv"
df_missing.sort_values(['APN','Year']).to_csv(missing_out, index=False)
print(f"Saved missing APNs → {missing_out}")

# Value mismatches
if len(df_diffs):
    diffs_out = out_dir / "ResUnits_value_mismatches.csv"
    df_diffs[['OID','APN','Year','Residential_Units_CSV','Residential_Units']].rename(
        columns={'Residential_Units':'Residential_Units_FC'}
    ).sort_values(['APN','Year']).to_csv(diffs_out, index=False)
    print(f"Saved mismatches     → {diffs_out}")
else:
    print("No value mismatches found — FC values already match CSV for all overlapping APN×Year rows.")

## 6. Update Residential_Units in Feature Class

**Run only after reviewing the validation above.**  
This updates `Residential_Units` for all APN×Year rows where the value differs.

In [ ]:
# Build lookup: OID → new value
if len(df_diffs) == 0:
    print("Nothing to update — all matched rows already have correct values.")
else:
    update_dict = dict(zip(df_diffs['OID'], df_diffs['Residential_Units_CSV']))
    print(f"Rows to update: {len(update_dict):,}")

    updated = 0
    with arcpy.da.UpdateCursor(fc_path, ['OID@', FC_RES_FIELD]) as cursor:
        for row in cursor:
            oid = row[0]
            if oid in update_dict:
                row[1] = update_dict[oid]
                cursor.updateRow(row)
                updated += 1

    print(f"Updated {updated:,} rows in {fc_path}")

## 7. Next Steps — Missing APNs Need Geometry

The APNs below exist in the CSV but have **no matching record** in `Parcel_History_Attributed`.  
We need to find their geometry from another source before we can insert new rows.

**Candidate geometry sources:**
- `sde.SDE.Parcels\sde.SDE.Parcel_Master` — current parcel boundaries (Vector.sde)
- `sde.SDE.Parcels\sde.SDE.Parcel_History` — historical parcel boundaries
- MapServer: https://maps.trpa.org/server/rest/services/ (browse for parcel layers)

Run the cell below to see the missing APNs and check if they exist in `Parcel_Master`.

In [ ]:
print(f"APNs missing from Parcel_History_Attributed ({len(missing_from_fc)}):")
for apn in sorted(missing_from_fc):
    print(f"  {apn}")

In [ ]:
# Check Parcel_Master for missing APNs
sde_parcel_master = str(sdeBase / "sde.SDE.Parcels" / "sde.SDE.Parcel_Master")

master_rows = []
with arcpy.da.SearchCursor(sde_parcel_master, ['APN', 'SHAPE@']) as cursor:
    for row in cursor:
        apn = str(row[0]).strip() if row[0] else None
        if apn in missing_from_fc:
            master_rows.append({'APN': apn, 'has_geometry': row[1] is not None})

df_master_found = pd.DataFrame(master_rows)

if len(df_master_found):
    found_in_master = set(df_master_found['APN'])
    still_missing   = missing_from_fc - found_in_master
    print(f"Found in Parcel_Master  : {len(found_in_master):,}")
    print(f"Not in Parcel_Master    : {len(still_missing):,} — need historical source")
    if still_missing:
        print("  APNs still unresolved:", sorted(still_missing))
    df_master_found
else:
    print("None of the missing APNs found in Parcel_Master.")
    print("Check Parcel_History or other sources.")

In [ ]:
# Check Parcel_History for APNs still unresolved after Parcel_Master check
# (Only run if 'still_missing' is populated above)

try:
    if len(still_missing) == 0:
        print("No APNs left to search in Parcel_History.")
    else:
        sde_parcel_history = str(sdeBase / "sde.SDE.Parcels" / "sde.SDE.Parcel_History")

        history_rows = []
        with arcpy.da.SearchCursor(sde_parcel_history, ['APN', 'YEAR', 'SHAPE@']) as cursor:
            for row in cursor:
                apn = str(row[0]).strip() if row[0] else None
                if apn in still_missing:
                    history_rows.append({
                        'APN' : apn,
                        'Year': row[1],
                        'has_geometry': row[2] is not None
                    })

        df_history_found = pd.DataFrame(history_rows)
        if len(df_history_found):
            print(f"Found in Parcel_History: {df_history_found['APN'].nunique():,} unique APNs")
            df_history_found.sort_values(['APN','Year'])
        else:
            print("Not found in Parcel_History either. May need to check other sources.")
except NameError:
    print("Run the Parcel_Master cell above first.")